# Mini Benchmark Evaluator — Kaggle demo

Tutorial **"The Science of Benchmarking — What's Measured, What's Missing, What's Next"** (NeurIPS 2025).

Two real benchmarks (**MMLU**, **GSM8K**) + **Gemma-4** models + non-LLM baselines, run through the tutorial's critique toolkit (saturation, baselines, metric brittleness, robustness).

### Kaggle setup (do this first)
1. **Settings → Accelerator → `GPU T4 x2`** (2×16 GB = 32 GB ⇒ even the 26B-A4B MoE fits).
2. **Settings → Internet → On** (needed for pip + Hugging Face download).
3. **Add-ons → Secrets →** add a secret named **`HF_TOKEN`** (a HF *read* token). Accept the Gemma licence once at <https://huggingface.co/google/gemma-4-E4B-it>.
4. **Run → Run all.** Greedy decoding ⇒ deterministic ⇒ reproducible.

## 1. Get the code + deps
Kaggle ships a CUDA build of `torch` already — do **not** reinstall it. We add only the missing libraries.

In [ ]:
# Option A: clone from GitHub (replace URL with this repo).
# Option B: attach the repo as a Kaggle Dataset and skip the clone
#           (it will be under /kaggle/input/<dataset-name>/).
REPO_URL = "https://github.com/ThuongHong/science-of-benchmarking-demo.git"

import os, sys, subprocess
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
WORK = f"/kaggle/working/{REPO_DIR}"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, "src")

# only the libs Kaggle doesn't already have; leave torch untouched
!pip install -q -U transformers accelerate bitsandbytes datasets sentencepiece
print("cwd:", os.getcwd())

## 2. Hugging Face auth (via Kaggle Secret `HF_TOKEN`)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
print("HF login OK")

## 3. GPU check
`T4 x2` shows two devices. `device_map="auto"` (used by GemmaRunner) will shard a big model across both.

In [ ]:
import torch
print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", torch.cuda.get_device_name(i))

## 4. Build benchmark subsets (downloads MMLU + GSM8K once)

In [ ]:
import config
from data import build_subsets
subs = build_subsets(config.N_MMLU, config.N_GSM8K, config.SEED)
print({k: len(v) for k, v in subs.items()})
subs["gsm8k"][0]

## 5. Choose systems
Default = E2B + E4B + baselines. With `T4 x2` you can add the **26B-A4B MoE** as a third, stronger rung — set `ADD_26B = True` to make the saturation/scaling story sharper.

In [ ]:
ADD_26B = False  # True needs the T4 x2 accelerator (OOMs a single GPU)
systems = list(config.GEMMA_SYSTEMS)
if ADD_26B:
    systems.append({"kind": "gemma", "name": "gemma-4-26b",
                     "model_id": "google/gemma-4-26B-A4B-it"})
systems += config.BASELINES
[s.get("name", s.get("baseline")) for s in systems]

## 6. Run the evaluation
First call downloads each model. Greedy generations are cached, so re-runs resume instantly.

In [ ]:
import evaluate
df = evaluate.run(systems)
df.to_csv("results/per_item.csv", index=False)
print(len(df), "rows -> results/per_item.csv")
df.head()

## 7. Analysis + figures

In [ ]:
import analysis
analysis.main()

In [ ]:
from IPython.display import Image, display
for fig in ["saturation", "metric_gap", "robustness"]:
    display(Image(f"results/figures/{fig}.png"))

## 8. Save outputs
Everything under the cloned repo's `results/` is downloadable from the notebook's **Output** tab. Commit those CSVs + PNGs back to the repo to fill the README results tables.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "results")
print("zipped -> /kaggle/working/results.zip")